<a href="https://colab.research.google.com/github/deetijasmitha/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deetijasmitha/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### My lane: Refresh / Content Opportunity Scoring

**Task type: Ranking / scoring**

The decision I want to improve is: **Which content pages should an editor review first for possible refresh?**

I will create a priority score for each content item and use that score to rank pages from highest to lowest review priority.

The output supports a real content action: an editor can start with the highest-priority pages instead of manually checking all pages.

This is a ranking/scoring problem because the goal is not only to predict whether a page is declining. The goal is to decide **which pages should be looked at first**. FlyRank's framing guide maps "Which ones first?" to ranking/scoring and recommends Precision@K as a suitable success metric.

A wrong ranking can waste editor time or cause an important page to be missed, so the ranking should prioritize pages with stronger evidence of an opportunity.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target / Proxy

The dataset does not contain a ready-made target label. I created a proxy target called is_declining_label using the trend_direction column.

-->1 = content is declining (trend_direction = "down")

-->0 = content is stable or improving

This is a proxy for identifying refresh opportunities, not a true future prediction.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Success metric: Precision@K

I will use **Precision@K** as the main success metric.

For example, Precision@50 asks:

**Of the 50 pages ranked highest for review, how many are actually marked as declining according to the proxy label?**

A higher Precision@50 means that the review queue contains a larger proportion of pages with the target opportunity signal.

This metric matches the real business action because editors have limited time and will usually review only the highest-priority pages first.

I chose Precision@K instead of accuracy because the goal is not to classify every page equally. The goal is to make the **top of the review queue useful**.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [22]:
# Create the proxy target from trend_direction
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("is_declining_label distribution:")
print(df["is_declining_label"].value_counts())

print("\nProportion:")
print(df["is_declining_label"].value_counts(normalize=True))

is_declining_label distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Proportion:
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


In [23]:
display(
    df[
        [
            "content_id",
            "content_type",
            "main_intent",
            "impressions_90d",
            "clicks_90d",
            "sessions_90d",
            "ctr",
            "avg_position",
            "engagement_rate",
            "is_declining_label",
        ]
    ].head(10)
)

,content_id,content_type,main_intent,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,engagement_rate,is_declining_label
0,content_304f48230142,keyword article,transactional,3803,29,17,0.76,10.6,5.88,1
1,content_a1fb4e703a9e,keyword article,informational,15320,7,9,0.05,20.3,0.00,1
2,content_9aa793d4d895,keyword article,informational,12581,11,11,0.09,36.5,0.00,1
3,content_331d6c4de07b,keyword article,commercial,11751,58,78,0.49,6.2,1.28,0
4,content_d99b7a2d90ca,keyword article,informational,19140,24,145,0.13,44.0,0.00,1
5,content_d4084a4bc775,keyword article,transactional,3970,1,5,0.03,8.5,0.00,1
6,content_9a34b442b552,keyword article,informational,20,0,1,0.00,7.0,0.00,1
7,content_a63219c6e95a,keyword article,commercial,1724,1,28,0.06,21.2,3.57,0
8,content_5e6c160719bc,keyword article,informational,32574,29,68,0.09,46.0,5.88,1
9,content_c27558df2b0c,keyword article,informational,1240,2,3,0.16,4.9,0.00,1


In [24]:
# Summary of the proxy target

target_summary = (
    df["is_declining_label"]
    .value_counts()
    .rename(index={0: "Not declining", 1: "Declining"})
    .to_frame("Count")
)

target_summary["Percentage"] = (
    target_summary["Count"] / len(df) * 100
).round(2)

target_summary

,Count,Percentage
is_declining_label,,
Declining,16262,54.21
Not declining,13738,45.79


In [25]:
# Check missing values in the main signals

missing = df[candidate_features].isna().sum().sort_values(ascending=False)

missing.to_frame("Missing values")

,Missing values
scroll_rate,125
clicks_90d,0
pageviews_90d,0
sessions_90d,0
impressions_90d,0
users_90d,0
engaged_sessions_90d,0
days_since_last_update,0
content_age_days,0
ctr,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why ML can beat a fixed rule

A fixed rule could use one threshold, such as reviewing every page with a low CTR or a high average search position. However, the dataset contains many different signals that describe content performance, including impressions, clicks, sessions, content age, days since the last update, CTR, average position, engagement rate, scroll rate, and AI traffic.

These signals have very different ranges and can interact with each other. For example, a page may have many impressions but very few clicks, while another page may have fewer impressions but stronger engagement. A single threshold would not capture all of these different situations.

A scoring model can combine multiple signals and learn which combinations are more useful for identifying pages with the refresh-opportunity proxy.

The output would support a real content action: **rank pages by refresh priority so that editors can review the highest-priority pages first.**

ML is useful here as decision support because it can handle multiple interacting signals more flexibly than a small set of manually chosen rules. It should not automatically decide that a page must be refreshed; the final decision remains with the content team.


In [26]:
candidate_features = [
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

display(df[candidate_features].describe().T)

,count,mean,std,min,25%,50%,75%,max
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.00,3615.25,517715.0
clicks_90d,30000.0,16.097333,75.076958,0.0,0.0,1.00,7.00,4178.0
pageviews_90d,30000.0,49.942467,152.101430,0.0,2.0,8.00,33.00,5998.0
sessions_90d,30000.0,37.066633,107.069131,1.0,2.0,7.00,27.00,4345.0
users_90d,30000.0,35.937700,103.748185,1.0,2.0,7.00,27.00,4913.0
engaged_sessions_90d,30000.0,0.991933,4.359576,0.0,0.0,0.00,1.00,290.0
content_age_days,30000.0,256.167800,132.707930,90.0,132.0,236.00,333.00,564.0
days_since_last_update,30000.0,46.098300,42.078709,1.0,20.0,20.00,104.00,373.0
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,100.0
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.80,22.30,245.0


scroll_rate       29,875 / 30,000

In [27]:
# Check missing values in the candidate signals

missing = df[candidate_features].isna().sum().sort_values(ascending=False)

display(
    missing[missing > 0].to_frame("Missing values")
)

,Missing values
scroll_rate,125


The candidate signals are not completely free of missing values. For example, `scroll_rate` has 125 missing values out of 30,000 content items. This is another reason a production ML workflow should explicitly handle missing data rather than automatically treating missing values as zero.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.